In [4]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
import pymupdf4llm
loaded_document=pymupdf4llm.to_markdown("D:/Chatbot/parent_retrival/ingestion/raw_data/finalHR.pdf")


Processing D:/Chatbot/parent_retrival/ingestion/raw_data/finalHR.pdf...
[                                        ] (0/13[                                        ] (  1/13[                                        ] (  2/13[                                        ] (  3/136[=                                       ] (  4/136[=                                       ] (  5/136[=                                       ] (  6/13[==                                      ] (  7/13[==                                      ] (  8/13[==                                      ] (  9/13[==                                      ] ( 10/136[===                                     ] ( 11/1[===                                     ] ( 12/1[===                                     ] ( 13/13[====                                    ] ( 14/13[====                                    ] ( 15/13[====                                    ] ( 16/136[=====                                   ] ( 17/1[=====                      

In [13]:
with open("man.md","w",encoding="utf-8") as f:
    f.write(loaded_document)


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
from langchain_chroma import Chroma

vector_store = Chroma(collection_name="HR_manual",embedding_function=embeddings,persist_directory="./RAG")

d:\Chatbot\parent_retrival\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_text_splitters import MarkdownHeaderTextSplitter,RecursiveCharacterTextSplitter
from langchain.storage._lc_store import create_kv_docstore
from langchain.storage import LocalFileStore
headers_to_split_on = [
    ("##", "Header 2"),
    ("###", "Header 3"),
    ("####", "Header 4"),
]

# create local_store
fs = LocalFileStore("./store_location")
store = create_kv_docstore(fs)

#split md loaded file, base on header
# markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on,strip_headers=False)
# md_header_splits = markdown_splitter.split_text(loaded_document)


# define child splitter
child_splitter=RecursiveCharacterTextSplitter(
    chunk_size=250,  # chunk size (characters)
    chunk_overlap=0,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
    length_function=len
)

# Define parent splitter
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)


In [5]:
from langchain.retrievers import ParentDocumentRetriever
retriever = ParentDocumentRetriever(
    vectorstore=vector_store,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

In [15]:
%pip install "unstructured[md]"

  Using cached Markdown-3.7-py3-none-any.whl.metadata (7.0 kB)
Using cached Markdown-3.7-py3-none-any.whl (106 kB)
Note: you may need to restart the kernel to use updated packages.


In [16]:
# loaded_document
from langchain_community.document_loaders import UnstructuredMarkdownLoader
loader = UnstructuredMarkdownLoader("man.md")
data = loader.load()


Document(metadata={'source': 'man.md'}, page_content='Executive Summary\n\n❖ The Executive Service Rules first introduced in May, 1981 and amended, as per BOD’s\n\napprovals from time to time are currently applicable.\n\n❖ On the directions of HR,R&NC / BOD, services of HR Consultant were commissioned\n\nto review and assess suitability of existing rules per current market conditions & propose amendments in policies, practices and document & prepare a new Manual.\n\n❖ Financial & Admin. Powers for each Position / Level have also been made an integral\n\npart of HR Manual, as per advice of Chairman, HR,R&NC.\n\n❖ The Board of Directors approved the HR Manual in its 344[th ] meeting held on 10-06-08.\n\nIn addition to amendment in different clauses from time to time, the HR Manual has been thoroughly reviewed, revised and approved by the Board of Directors in its 485[th ] meeting held on 18-03-2017 and 587[th ] meeting held on 21-06-2022.\n\n❖ Following Topics are covered by the HR Manua

In [10]:
retriever.add_documents(loaded_document)

AttributeError: 'str' object has no attribute 'page_content'